In [ ]:
# Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Any, Optional
from pathlib import Path

# Configure plotting
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

print("✅ Libraries loaded")

In [ ]:
# Load sample data
titanic = pd.read_csv("../data/titanic.csv")
print(f"📊 Loaded Titanic: {titanic.shape}")
titanic.head()

## 1. EDA Analyzer Class

In [ ]:
class EDAAnalyzer:
    """
    Comprehensive EDA toolkit for data analysis.
    """
    
    def __init__(self, df: pd.DataFrame, name: str = "Dataset"):
        self.df = df
        self.name = name
        self.numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        self.categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    
    def overview(self) -> Dict[str, Any]:
        """
        Get basic dataset overview.
        """
        return {
            'name': self.name,
            'rows': len(self.df),
            'columns': len(self.df.columns),
            'numeric_cols': len(self.numeric_cols),
            'categorical_cols': len(self.categorical_cols),
            'missing_cells': self.df.isna().sum().sum(),
            'missing_pct': f"{self.df.isna().sum().sum() / (self.df.shape[0] * self.df.shape[1]) * 100:.2f}%",
            'duplicates': self.df.duplicated().sum(),
            'memory_mb': f"{self.df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
        }
    
    def print_overview(self):
        """
        Print formatted overview.
        """
        info = self.overview()
        print("\n" + "="*50)
        print(f"📊 DATASET OVERVIEW: {info['name']}")
        print("="*50)
        print(f"📐 Shape: {info['rows']} rows × {info['columns']} columns")
        print(f"🔢 Numeric columns: {info['numeric_cols']}")
        print(f"📝 Categorical columns: {info['categorical_cols']}")
        print(f"❓ Missing values: {info['missing_cells']} ({info['missing_pct']})")
        print(f"🔄 Duplicates: {info['duplicates']}")
        print(f"💾 Memory: {info['memory_mb']}")

In [ ]:
# Create analyzer and show overview
eda = EDAAnalyzer(titanic, "Titanic")
eda.print_overview()

## 2. Missing Value Analysis

In [ ]:
class MissingValueAnalyzer:
    """
    Analyze and visualize missing values.
    """
    
    @staticmethod
    def analyze(df: pd.DataFrame) -> pd.DataFrame:
        """
        Get detailed missing value report.
        """
        missing = df.isna().sum()
        missing_pct = df.isna().mean() * 100
        
        report = pd.DataFrame({
            'Missing Count': missing,
            'Missing %': missing_pct.round(2),
            'Dtype': df.dtypes
        })
        
        return report[report['Missing Count'] > 0].sort_values('Missing %', ascending=False)
    
    @staticmethod
    def plot_missing(df: pd.DataFrame, figsize=(12, 5)):
        """
        Visualize missing values.
        """
        missing_pct = df.isna().mean() * 100
        missing_pct = missing_pct[missing_pct > 0].sort_values(ascending=False)
        
        if len(missing_pct) == 0:
            print("✅ No missing values found!")
            return
        
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        
        # Bar plot
        colors = ['#ff6b6b' if x > 50 else '#ffd93d' if x > 20 else '#6bcb77' for x in missing_pct]
        missing_pct.plot(kind='barh', ax=axes[0], color=colors)
        axes[0].set_xlabel('Missing %')
        axes[0].set_title('Missing Values by Column')
        axes[0].axvline(x=50, color='red', linestyle='--', alpha=0.5, label='50% threshold')
        
        # Heatmap
        missing_matrix = df[missing_pct.index].isna().T
        sns.heatmap(missing_matrix, cbar=False, ax=axes[1], cmap='Reds', yticklabels=True)
        axes[1].set_title('Missing Value Patterns')
        axes[1].set_xlabel('Sample Index')
        
        plt.tight_layout()
        plt.show()

In [ ]:
# Analyze missing values
print("❓ Missing Value Report:")
MissingValueAnalyzer.analyze(titanic)

In [ ]:
# Visualize missing values
MissingValueAnalyzer.plot_missing(titanic)

## 3. Statistical Analysis

In [ ]:
class StatisticalAnalyzer:
    """
    Perform statistical analysis on DataFrame.
    """
    
    @staticmethod
    def numeric_stats(df: pd.DataFrame) -> pd.DataFrame:
        """
        Get comprehensive statistics for numeric columns.
        """
        numeric_df = df.select_dtypes(include=[np.number])
        
        stats = numeric_df.describe().T
        stats['median'] = numeric_df.median()
        stats['skew'] = numeric_df.skew()
        stats['kurtosis'] = numeric_df.kurtosis()
        stats['missing'] = numeric_df.isna().sum()
        stats['unique'] = numeric_df.nunique()
        
        return stats.round(3)
    
    @staticmethod
    def categorical_stats(df: pd.DataFrame) -> pd.DataFrame:
        """
        Get statistics for categorical columns.
        """
        cat_df = df.select_dtypes(include=['object', 'category'])
        
        stats = []
        for col in cat_df.columns:
            stats.append({
                'column': col,
                'unique': cat_df[col].nunique(),
                'top': cat_df[col].mode().iloc[0] if len(cat_df[col].mode()) > 0 else None,
                'top_freq': cat_df[col].value_counts().iloc[0] if len(cat_df[col].value_counts()) > 0 else 0,
                'missing': cat_df[col].isna().sum(),
                'missing_pct': f"{cat_df[col].isna().mean()*100:.1f}%"
            })
        
        return pd.DataFrame(stats).set_index('column')

In [ ]:
# Numeric statistics
print("🔢 Numeric Column Statistics:")
StatisticalAnalyzer.numeric_stats(titanic)

In [ ]:
# Categorical statistics
print("📝 Categorical Column Statistics:")
StatisticalAnalyzer.categorical_stats(titanic)

## 4. Correlation Analysis

In [ ]:
class CorrelationAnalyzer:
    """
    Analyze correlations between variables.
    """
    
    @staticmethod
    def compute_correlation(df: pd.DataFrame, method: str = 'pearson') -> pd.DataFrame:
        """
        Compute correlation matrix.
        
        Args:
            df: DataFrame
            method: 'pearson', 'spearman', or 'kendall'
        """
        numeric_df = df.select_dtypes(include=[np.number])
        return numeric_df.corr(method=method).round(3)
    
    @staticmethod
    def plot_correlation(df: pd.DataFrame, method: str = 'pearson', figsize=(10, 8)):
        """
        Plot correlation heatmap.
        """
        corr = CorrelationAnalyzer.compute_correlation(df, method)
        
        plt.figure(figsize=figsize)
        mask = np.triu(np.ones_like(corr, dtype=bool))
        
        sns.heatmap(
            corr,
            mask=mask,
            annot=True,
            fmt='.2f',
            cmap='RdBu_r',
            center=0,
            square=True,
            linewidths=0.5
        )
        plt.title(f'Correlation Matrix ({method.title()})')
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def find_high_correlations(df: pd.DataFrame, threshold: float = 0.7) -> pd.DataFrame:
        """
        Find pairs of highly correlated features.
        """
        corr = CorrelationAnalyzer.compute_correlation(df)
        
        high_corr = []
        for i in range(len(corr.columns)):
            for j in range(i+1, len(corr.columns)):
                if abs(corr.iloc[i, j]) >= threshold:
                    high_corr.append({
                        'Feature 1': corr.columns[i],
                        'Feature 2': corr.columns[j],
                        'Correlation': corr.iloc[i, j]
                    })
        
        return pd.DataFrame(high_corr).sort_values('Correlation', ascending=False, key=abs)

In [ ]:
# Correlation heatmap
CorrelationAnalyzer.plot_correlation(titanic)

In [ ]:
# Find high correlations
print("🔗 High Correlations (threshold=0.3):")
CorrelationAnalyzer.find_high_correlations(titanic, threshold=0.3)

## 5. Distribution Analysis

In [ ]:
class DistributionAnalyzer:
    """
    Analyze and visualize data distributions.
    """
    
    @staticmethod
    def plot_numeric_distributions(df: pd.DataFrame, cols: Optional[List[str]] = None, figsize=(15, 4)):
        """
        Plot distributions for numeric columns.
        """
        if cols is None:
            cols = df.select_dtypes(include=[np.number]).columns.tolist()[:6]
        
        n_cols = min(len(cols), 4)
        n_rows = (len(cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows))
        axes = axes.flatten() if n_rows * n_cols > 1 else [axes]
        
        for idx, col in enumerate(cols):
            ax = axes[idx]
            df[col].hist(bins=30, ax=ax, edgecolor='black', alpha=0.7)
            ax.axvline(df[col].mean(), color='red', linestyle='--', label=f'Mean: {df[col].mean():.2f}')
            ax.axvline(df[col].median(), color='green', linestyle='--', label=f'Median: {df[col].median():.2f}')
            ax.set_title(f'{col}')
            ax.legend(fontsize=8)
        
        # Hide empty axes
        for idx in range(len(cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('Numeric Distributions', y=1.02)
        plt.tight_layout()
        plt.show()
    
    @staticmethod
    def plot_categorical_distributions(df: pd.DataFrame, cols: Optional[List[str]] = None, figsize=(15, 4)):
        """
        Plot distributions for categorical columns.
        """
        if cols is None:
            cols = df.select_dtypes(include=['object', 'category']).columns.tolist()[:4]
        
        n_cols = min(len(cols), 4)
        n_rows = (len(cols) + n_cols - 1) // n_cols
        
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows))
        if n_rows * n_cols == 1:
            axes = [axes]
        else:
            axes = axes.flatten()
        
        for idx, col in enumerate(cols):
            ax = axes[idx]
            value_counts = df[col].value_counts().head(10)
            value_counts.plot(kind='bar', ax=ax, edgecolor='black')
            ax.set_title(f'{col}')
            ax.tick_params(axis='x', rotation=45)
        
        # Hide empty axes
        for idx in range(len(cols), len(axes)):
            axes[idx].set_visible(False)
        
        plt.suptitle('Categorical Distributions', y=1.02)
        plt.tight_layout()
        plt.show()

In [ ]:
# Plot numeric distributions
DistributionAnalyzer.plot_numeric_distributions(titanic, ['Age', 'Fare', 'Pclass', 'SibSp'])

In [ ]:
# Plot categorical distributions
DistributionAnalyzer.plot_categorical_distributions(titanic, ['Sex', 'Embarked'])

## 6. Outlier Detection

In [ ]:
class OutlierDetector:
    """
    Detect and analyze outliers.
    """
    
    @staticmethod
    def detect_iqr(df: pd.DataFrame, column: str, multiplier: float = 1.5) -> pd.DataFrame:
        """
        Detect outliers using IQR method.
        """
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        
        lower_bound = Q1 - multiplier * IQR
        upper_bound = Q3 + multiplier * IQR
        
        outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
        
        return {
            'column': column,
            'Q1': Q1,
            'Q3': Q3,
            'IQR': IQR,
            'lower_bound': lower_bound,
            'upper_bound': upper_bound,
            'outlier_count': len(outliers),
            'outlier_pct': f"{len(outliers) / len(df) * 100:.2f}%",
            'outlier_indices': outliers.index.tolist()
        }
    
    @staticmethod
    def detect_zscore(df: pd.DataFrame, column: str, threshold: float = 3) -> Dict:
        """
        Detect outliers using Z-score method.
        """
        z_scores = np.abs((df[column] - df[column].mean()) / df[column].std())
        outliers = df[z_scores > threshold]
        
        return {
            'column': column,
            'threshold': threshold,
            'outlier_count': len(outliers),
            'outlier_pct': f"{len(outliers) / len(df) * 100:.2f}%",
            'outlier_indices': outliers.index.tolist()
        }
    
    @staticmethod
    def plot_outliers(df: pd.DataFrame, cols: Optional[List[str]] = None, figsize=(12, 4)):
        """
        Box plots to visualize outliers.
        """
        if cols is None:
            cols = df.select_dtypes(include=[np.number]).columns.tolist()[:5]
        
        plt.figure(figsize=figsize)
        df[cols].boxplot()
        plt.title('Box Plots - Outlier Visualization')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.show()

In [ ]:
# Detect outliers in Age column using IQR
outlier_info = OutlierDetector.detect_iqr(titanic.dropna(subset=['Age']), 'Age')

print("🎯 Outlier Detection (IQR Method) - Age:")
for key, value in outlier_info.items():
    if key != 'outlier_indices':
        print(f"   {key}: {value}")

In [ ]:
# Visualize outliers with box plots
OutlierDetector.plot_outliers(titanic, ['Age', 'Fare', 'SibSp', 'Parch'])

## 7. Complete EDA Report

In [ ]:
def generate_eda_report(df: pd.DataFrame, name: str = "Dataset") -> Dict[str, Any]:
    """
    Generate a complete EDA report.
    """
    print(f"\n{'='*60}")
    print(f"📊 COMPLETE EDA REPORT: {name}")
    print(f"{'='*60}")
    
    # 1. Overview
    eda = EDAAnalyzer(df, name)
    eda.print_overview()
    
    # 2. Missing Values
    print("\n" + "-"*40)
    print("❓ MISSING VALUES")
    print("-"*40)
    missing_report = MissingValueAnalyzer.analyze(df)
    if len(missing_report) > 0:
        print(missing_report)
    else:
        print("✅ No missing values!")
    
    # 3. Numeric Statistics
    print("\n" + "-"*40)
    print("🔢 NUMERIC STATISTICS")
    print("-"*40)
    print(StatisticalAnalyzer.numeric_stats(df))
    
    # 4. High Correlations
    print("\n" + "-"*40)
    print("🔗 HIGH CORRELATIONS")
    print("-"*40)
    high_corr = CorrelationAnalyzer.find_high_correlations(df, threshold=0.3)
    if len(high_corr) > 0:
        print(high_corr)
    else:
        print("No correlations above threshold")
    
    # 5. Outlier Summary
    print("\n" + "-"*40)
    print("🎯 OUTLIER SUMMARY")
    print("-"*40)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df[col].isna().all():
            continue
        result = OutlierDetector.detect_iqr(df.dropna(subset=[col]), col)
        if result['outlier_count'] > 0:
            print(f"   {col}: {result['outlier_count']} outliers ({result['outlier_pct']})")
    
    print(f"\n{'='*60}")
    print("📊 EDA Report Complete")
    print(f"{'='*60}")
    
    return {
        'overview': eda.overview(),
        'missing': missing_report.to_dict() if len(missing_report) > 0 else {},
        'correlations': high_corr.to_dict() if len(high_corr) > 0 else {}
    }

In [ ]:
# Generate complete EDA report
report = generate_eda_report(titanic, "Titanic")

## ✅ Summary

This EDA module provides:

**1. EDAAnalyzer**
- Dataset overview with shape, types, memory
- Automatic column type detection

**2. MissingValueAnalyzer**
- Missing value report
- Visual missing value patterns

**3. StatisticalAnalyzer**
- Comprehensive numeric statistics
- Categorical column analysis

**4. CorrelationAnalyzer**
- Correlation matrix computation
- Heatmap visualization
- High correlation detection

**5. DistributionAnalyzer**
- Numeric distribution plots
- Categorical frequency plots

**6. OutlierDetector**
- IQR-based detection
- Z-score detection
- Box plot visualization